# 14 · Inventory policy engine (Step 7 — first planning engine)

Turns detected inventory abnormalities into **quantified replenishment recommendations**. Computes, per
stocked **material × plant** with **independent (sales) demand**, an order-up-to policy — protection
interval `P`, expected lead-time demand `μ_X`, safety stock `SS`, order-up-to level `S*`, suggested
order quantity — and writes a `recommended_action` onto the open inventory events.

**Method (v1, distribution-free / conformal):** `S* = μ_X + SS`, where `μ_X` is the point forecast
summed over `P = L + R` and `SS = max(0, Q̂_SL(e_P))` is the **split-conformal quantile** of the
cumulative `P`-step forecast errors (`monitoring.demand_forecast_error_sets`, from nb `02`; `14` selects
`h=P`). Only point forecasts are needed; correct for skewed demand. Full method + assumptions:
**`inventory_policy_model.html`**.

## Scope — by demand independence (not by purchased-vs-made)
Reorder-point / order-up-to is an **independent-demand** technique, so this engine covers **items that
have a sales demand forecast**. How the target is *replenished* then depends on how the item is sourced:

| Item class | In scope? | `replenishment_mode` | Lead time source |
|---|---|---|---|
| Independent demand, **purchased** (e.g. green beans sold bulk) | yes | `PO` | procurement twin (supplier) |
| Independent demand, **produced** (e.g. roasted 500g finished goods) | yes | `PRODUCTION` (order-up-to = MPS signal handed to MRP/scheduling) | production twin (actual) |
| **Dependent demand** only — components/intermediates consumed in production (packaging, dairy, bulk) | no | `MRP_DEPENDENT` | — (netted by MRP via BOM explosion) |

Dependent-demand items are **not** given a naive reorder point — their demand is derived by MRP from the
finished-goods plan (next engine). For an item with **both** independent and dependent demand (green
beans are sold *and* roasted), this engine covers only the **independent (sales)** portion; MRP adds the
dependent portion.

**Reads:** `inventory_twin_current` (position), `procurement_twin_current` (supplier lead time, open PO,
primary supplier), `production_twin_current` (production lead time, open production qty),
`demand_twin_metrics_timeseries` (forecast → `μ_X`), `demand_forecast_error_sets` (conformal calibration),
`planning_config` (SL, review period, MOQ, ...). **Writes:** `monitoring.inventory_policy` (deterministic
overwrite) + `recommended_action` on open `INVENTORY` events (run after `12`).

## Config

In [ ]:
dbutils.widgets.text("target_catalog", "mickmickstableawscoff_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"

import math, uuid
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

REFRESH_ID = str(uuid.uuid4())
print("inventory policy target:", f"{FQ_MON}.inventory_policy")

## Policy parameters (from `planning_config` via `cfg()`)
Same `cfg()` helper as the twins/events — reads an effective GLOBAL value with a safe fallback.

In [ ]:
def cfg(key, default, kind="double"):
    col = "value_string" if kind == "string" else "value_double"
    try:
        r = spark.sql(f"SELECT {col} FROM {FQ_MON}.v_resolved_config WHERE param_key = '{key}'").collect()
        return r[0][0] if r and r[0][0] is not None else default
    except Exception:
        return default

SL = float(cfg("service_level", 0.95))
R_DAYS = float(cfg("review_period_days", 7.0))
MOQ = float(cfg("min_order_qty", 0.0))
MULT = float(cfg("order_multiple", 1.0))
R_WEEKS = max(1, int(round(R_DAYS / 7.0)))   # review period in forecast buckets (weeks)
print(f"SL={SL}  R={R_DAYS}d ({R_WEEKS}w)  MOQ={MOQ}  order_multiple={MULT}")

## Assemble inputs
Small data — pull to the driver as pandas and compute the policy in plain Python. Cumulative forecast
errors **add across channels**, so the demand series (channel grain) aggregate cleanly to the inventory
grain (material × plant). Independent demand comes from the sales forecast; sourcing (purchased vs
produced) comes from the procurement / production twins.

In [ ]:
df_inv = spark.table(f"{FQ_MON}.inventory_twin_current").toPandas()
df_proc = spark.table(f"{FQ_MON}.procurement_twin_current").toPandas()
df_prod = spark.table(f"{FQ_MON}.production_twin_current").toPandas()
df_fc = (spark.table(f"{FQ_MON}.demand_twin_metrics_timeseries")
         .where("period_type = 'FORECAST'")
         .select("material", "plant", "period_start", "forecast_qty").toPandas())
try:
    df_err = spark.table(f"{FQ_MON}.demand_forecast_error_sets").toPandas()
except Exception:
    df_err = pd.DataFrame(columns=["material", "plant", "channel", "horizon",
                                   "origin_index", "cum_error", "cum_forecast"])

# Purchased sourcing: primary supplier per material x plant = largest PO volume (STRONG ASSUMPTION:
# single-source; ignores dual-sourcing / cheaper-or-faster alternates). Open PO summed across suppliers.
_ps = df_proc.sort_values("po_qty", ascending=False).groupby(["material", "plant"], as_index=False).first()
primary = _ps[["material", "plant", "supplier", "supplier_name", "avg_lead_time_days"]]
openpo = df_proc.groupby(["material", "plant"], as_index=False)["open_po_qty"].sum()

# Produced sourcing: production lead time (recorded actual) + open production (WIP) per material x plant.
prod = df_prod[["material", "plant", "avg_actual_lead_time_hours", "open_production_qty"]].copy()

# Independent demand: sales forecast summed across channels, ranked by period (mu_X = sum over first P).
fc_mp = (df_fc.groupby(["material", "plant", "period_start"], as_index=False)["forecast_qty"].sum()
              .sort_values(["material", "plant", "period_start"]))
fc_mp["rank"] = fc_mp.groupby(["material", "plant"]).cumcount() + 1

# Error sets aggregated to material x plant (cumulative errors add across channels).
if len(df_err):
    err_mp = (df_err.groupby(["material", "plant", "horizon", "origin_index"], as_index=False)
                    .agg(cum_error=("cum_error", "sum"), cum_forecast=("cum_forecast", "sum")))
else:
    err_mp = pd.DataFrame(columns=["material", "plant", "horizon", "origin_index", "cum_error", "cum_forecast"])

has_demand = set(map(tuple, fc_mp[["material", "plant"]].drop_duplicates().values.tolist()))
print(f"inventory positions={len(df_inv)}  with independent demand={len(has_demand)}  "
      f"purchased={len(primary)}  produced={len(prod)}")

## Compute the policy
Classify each stocked material × plant, then for **independent-demand** items build the order-up-to
policy; **dependent-only** items are flagged for MRP (no reorder point).

In [ ]:
def num(x, default=0.0):
    """Safe float: map None / NaN to a default (pass default=None to keep it nullable)."""
    if x is None:
        return default
    try:
        v = float(x)
    except (TypeError, ValueError):
        return default
    return default if (isinstance(v, float) and math.isnan(v)) else v

def conformal_q(errs, sl):
    """Split-conformal one-sided quantile: the ceil((n+1)*sl)-th order statistic (capped at max)."""
    e = np.sort(np.asarray(list(errs), dtype=float))
    n = int(e.size)
    if n == 0:
        return None, 0, True
    k = math.ceil((n + 1) * sl)
    return float(e[min(k, n) - 1]), n, (k > n)

def mu_over_P(m, p, P):
    s = fc_mp[(fc_mp.material == m) & (fc_mp.plant == p) & (fc_mp["rank"] <= P)]["forecast_qty"]
    return float(s.sum()) if len(s) else 0.0

def round_mult(q, mult):
    return round(q / mult) * mult if mult and mult > 0 else q

prod_lead = {(r.material, r.plant): num(r.avg_actual_lead_time_hours, None) for r in prod.itertuples()}
prod_open = {(r.material, r.plant): num(r.open_production_qty) for r in prod.itertuples()}

rows = []
for _, r in df_inv.iterrows():
    m, p, mg = r["material"], r["plant"], r["material_group"]
    uom, mtext = r["uom"], r["material_text"]
    on_hand = num(r["on_hand_qty"])
    doc = num(r["days_of_cover_actual"], None)
    key = (m, p)
    prow = primary[(primary.material == m) & (primary.plant == p)]
    has_proc = len(prow) > 0
    has_prod = key in prod_lead
    independent = key in has_demand

    base = dict(material=m, plant=p, material_text=mtext, material_group=mg, uom=uom,
                on_hand_qty=on_hand, days_of_cover_actual=doc,
                is_low_cover=bool(r["is_low_cover"]), is_stockout=bool(r["is_stockout"]),
                is_overstock=bool(r["is_overstock"]),
                primary_supplier=None, primary_supplier_name=None)

    # --- Dependent-demand items: not a reorder point; planned by MRP -----------------
    if not independent:
        note = ("Dependent demand - planned by MRP (net requirements from a BOM explosion of the "
                "finished-goods plan), not a reorder point. Pending the MRP engine.")
        rows.append({**base, "in_scope": False, "replenishment_mode": "MRP_DEPENDENT",
                     "lead_time_days": None, "lead_time_source": None, "protection_weeks": None,
                     "mu_lead_time_demand": None, "demand_source": "DEPENDENT",
                     "on_order_qty": 0.0, "inventory_position": on_hand, "safety_stock": None,
                     "order_up_to_level": None, "suggested_order_qty": None, "service_level": SL,
                     "calibration_method": "NA_DEPENDENT", "n_calibration": 0,
                     "insufficient_calibration": None,
                     "action_reorder": note, "action_overstock": note, "action_generic": note})
        continue

    # --- Independent-demand items: order-up-to policy --------------------------------
    if has_prod:
        mode = "PRODUCTION"; lt_days = prod_lead[key]; lt_src = "PRODUCTION"
        on_order = prod_open.get(key, 0.0)
        supplier = supplier_name = None
        lt_days = None if lt_days is None else lt_days / 24.0     # hours -> days
    elif has_proc:
        mode = "PO"; lt_days = num(prow["avg_lead_time_days"].iloc[0], None); lt_src = "SUPPLIER"
        opo = openpo[(openpo.material == m) & (openpo.plant == p)]["open_po_qty"]
        on_order = num(opo.iloc[0]) if len(opo) else 0.0
        supplier = prow["supplier"].iloc[0]; supplier_name = prow["supplier_name"].iloc[0]
    else:
        mode = "PO"; lt_days = None; lt_src = "DEFAULT"; on_order = 0.0
        supplier = supplier_name = None

    L = max(1, math.ceil((lt_days if lt_days is not None else 7.0) / 7.0))
    P = L + R_WEEKS
    position = on_hand + on_order
    mu = mu_over_P(m, p, P)

    es = err_mp[(err_mp.material == m) & (err_mp.plant == p) & (err_mp.horizon == P)]["cum_error"]
    if len(es) > 0:
        q, n_cal, insuf = conformal_q(es, SL); ss = max(0.0, num(q)); calib = "PER_SERIES_CONFORMAL"
    else:
        pool = err_mp[(err_mp.horizon == P) & (err_mp.cum_forecast > 0)]
        rel = (pool["cum_error"] / pool["cum_forecast"]).to_numpy()
        rq, n_cal, insuf = conformal_q(rel, SL)
        ss = max(0.0, num(rq) * mu); calib = "POOLED_RELATIVE"; insuf = True if rq is None else insuf

    s_star = mu + ss
    q_raw = s_star - position
    q_sug = max(round_mult(q_raw, MULT), MOQ) if q_raw > 0 else 0.0
    slp = f"{SL:.0%}"; cnote = "insufficient calibration; " if insuf else ""

    if mode == "PRODUCTION":
        target = "production order"; who = "production/MRP scheduling"
    else:
        target = "PO"; who = (supplier_name or supplier)
    if q_sug > 0:
        a_reorder = (f"Below target: position {position:,.0f} < order-up-to S*={s_star:,.0f} "
                     f"(forecast demand {mu:,.0f} + safety {ss:,.0f} over P={P}w). "
                     f"Suggest {target} of {q_sug:,.0f} {uom}"
                     + (f" to {who}" if mode == 'PO' else f" ({who})") + f" [{cnote}SL {slp}].")
    else:
        a_reorder = (f"OK: position {position:,.0f} >= order-up-to S*={s_star:,.0f} "
                     f"(P={P}w, SL {slp}); no {target} needed.")
    a_over = (f"Overstock: position {position:,.0f} exceeds order-up-to S*={s_star:,.0f} by "
              f"{position - s_star:,.0f} {uom}" + (f" ({doc:,.0f}d cover)" if doc is not None else "")
              + (f". Defer/reduce {target}" + (f" to {who}" if mode == 'PO' else "") + "."))
    a_generic = (f"Inventory policy for {mtext}: S*={s_star:,.0f} {uom} (P={P}w, SL {slp}, "
                 f"position {position:,.0f}); suggested {target} {q_sug:,.0f}.")

    rows.append({**base, "in_scope": True, "replenishment_mode": mode,
                 "primary_supplier": supplier, "primary_supplier_name": supplier_name,
                 "lead_time_days": lt_days, "lead_time_source": lt_src, "protection_weeks": P,
                 "mu_lead_time_demand": mu, "demand_source": "INDEPENDENT_FORECAST",
                 "on_order_qty": on_order, "inventory_position": position, "safety_stock": ss,
                 "order_up_to_level": s_star, "suggested_order_qty": q_sug, "service_level": SL,
                 "calibration_method": calib, "n_calibration": int(n_cal),
                 "insufficient_calibration": bool(insuf),
                 "action_reorder": a_reorder, "action_overstock": a_over, "action_generic": a_generic})

pol = pd.DataFrame(rows)
print("policy rows:", len(pol))
print(pol.groupby(["replenishment_mode"]).size().to_string())

## Write `monitoring.inventory_policy` (deterministic overwrite, typed schema)

In [ ]:
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                                   IntegerType, BooleanType)

SCHEMA = StructType([
    StructField("material", StringType()), StructField("plant", StringType()),
    StructField("material_text", StringType()), StructField("material_group", StringType()),
    StructField("uom", StringType()), StructField("in_scope", BooleanType()),
    StructField("replenishment_mode", StringType()), StructField("primary_supplier", StringType()),
    StructField("primary_supplier_name", StringType()), StructField("lead_time_days", DoubleType()),
    StructField("lead_time_source", StringType()), StructField("protection_weeks", IntegerType()),
    StructField("mu_lead_time_demand", DoubleType()), StructField("demand_source", StringType()),
    StructField("on_hand_qty", DoubleType()), StructField("on_order_qty", DoubleType()),
    StructField("inventory_position", DoubleType()), StructField("safety_stock", DoubleType()),
    StructField("order_up_to_level", DoubleType()), StructField("suggested_order_qty", DoubleType()),
    StructField("service_level", DoubleType()), StructField("calibration_method", StringType()),
    StructField("n_calibration", IntegerType()), StructField("insufficient_calibration", BooleanType()),
    StructField("days_of_cover_actual", DoubleType()), StructField("is_low_cover", BooleanType()),
    StructField("is_stockout", BooleanType()), StructField("is_overstock", BooleanType()),
    StructField("action_reorder", StringType()), StructField("action_overstock", StringType()),
    StructField("action_generic", StringType()),
])
ORDER = [f.name for f in SCHEMA.fields]
INT_COLS = {"protection_weeks", "n_calibration"}
BOOL_COLS = {"in_scope", "insufficient_calibration", "is_low_cover", "is_stockout", "is_overstock"}
STR_COLS = {"material", "plant", "material_text", "material_group", "uom", "replenishment_mode",
            "primary_supplier", "primary_supplier_name", "lead_time_source", "demand_source",
            "calibration_method", "action_reorder", "action_overstock", "action_generic"}

def _norm(col, v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    if col in INT_COLS:  return int(v)
    if col in BOOL_COLS: return bool(v)
    if col in STR_COLS:  return str(v)
    return float(v)

recs = [tuple(_norm(c, row.get(c)) for c in ORDER) for row in pol.to_dict("records")]
out = (spark.createDataFrame(recs, schema=SCHEMA)
       .withColumn("refresh_id", F.lit(REFRESH_ID))
       .withColumn("generated_at", F.current_timestamp()))
(out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ_MON}.inventory_policy"))
spark.sql(f"ALTER TABLE {FQ_MON}.inventory_policy SET TBLPROPERTIES ('comment' = "
          "'Inventory order-up-to policy per material x plant, scoped to independent (sales) demand. "
          "v1 = point forecast + split-conformal residual quantile; dependent-demand items flagged MRP_DEPENDENT. "
          "See inventory_policy_model.html.')")
print("wrote", spark.table(f"{FQ_MON}.inventory_policy").count(), "policy rows to", f"{FQ_MON}.inventory_policy")

## Write recommendations back onto open inventory events
Idempotent `MERGE`: fills `recommended_action` on open `INVENTORY` events by material × plant, choosing
the message by event type. Re-running `12` resets these, so run `14` after `12`.

In [ ]:
spark.sql(f"""
MERGE INTO {FQ_MON}.events e
USING {FQ_MON}.inventory_policy r
  ON e.function = 'INVENTORY' AND e.status = 'open'
     AND e.entity_id = r.material AND e.plant = r.plant
WHEN MATCHED THEN UPDATE SET e.recommended_action = CASE e.event_type
    WHEN 'STOCKOUT'  THEN r.action_reorder
    WHEN 'LOW_COVER' THEN r.action_reorder
    WHEN 'OVERSTOCK' THEN r.action_overstock
    ELSE r.action_generic END
""")
upd = spark.sql(f"SELECT COUNT(*) c FROM {FQ_MON}.events "
                f"WHERE function = 'INVENTORY' AND recommended_action IS NOT NULL").collect()[0]["c"]
print("inventory events annotated with recommended_action:", upd)

## Verify

In [ ]:
display(spark.sql(f"""
SELECT in_scope, replenishment_mode, demand_source, material_text, lead_time_source,
       protection_weeks, ROUND(mu_lead_time_demand, 0) AS mu, ROUND(safety_stock, 0) AS ss,
       ROUND(order_up_to_level, 0) AS s_star, ROUND(inventory_position, 0) AS position,
       ROUND(suggested_order_qty, 0) AS order_qty, calibration_method, n_calibration
FROM {FQ_MON}.inventory_policy
ORDER BY in_scope DESC, replenishment_mode, material
"""))
display(spark.sql(f"""
SELECT event_type, entity_label, plant, recommended_action
FROM {FQ_MON}.events WHERE function = 'INVENTORY' ORDER BY event_type, entity_label
"""))

## Next
Scoped to independent (sales) demand: purchased sold items → PO targets; produced sold items →
order-up-to as an **MPS signal**. The dependent-demand items flagged `MRP_DEPENDENT` (packaging, dairy,
roasted bulk) are the input to the **MRP engine** (next): explode the finished-goods order-up-to plan
through `gold.billofmaterials` into net component / raw-material requirements and planned orders. Then
finite production scheduling, distribution planning, and (Step 8) the Lakebase plan-of-record.